# Gen-SHM: Physics-Informed Generative Surrogate Demo

This notebook demonstrates the complete Gen-SHM workflow for drone wing structural health monitoring.

In [ ]:
# Import required libraries
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

# Import Gen-SHM modules
from models.surrogate_model import DroneWingSurrogate, quick_train_and_generate
from evaluation.visualization import SHMVisualizer
from evaluation.metrics import SHMMetrics
from utils.helpers import set_seed

# Set plotting style
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Quick Start: Generate Samples with Pre-trained Model

Let's start by generating synthetic vibration data for a damaged wing scenario.

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Generate samples with 20% damage at wing root
print("Generating synthetic vibration data...")
samples = quick_train_and_generate(
    damage_level=0.2,      # 20% stiffness reduction
    damage_location=0.0,   # Wing root (normalized position)
    num_samples=25         # Generate 25 independent samples
)

print(f"Generated {samples['acceleration'].shape[0]} samples")
print(f"Each sample has {samples['acceleration'].shape[1]} sensors")
print(f"Time series length: {samples['acceleration'].shape[2]} points")

In [ ]:
# Visualize the generated samples
visualizer = SHMVisualizer()

# Plot acceleration signals from first sample
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

time = samples['time']
acceleration = samples['acceleration'][0]  # First sample
sensor_positions = samples['sensor_positions']

for i, (sensor_pos, signal) in enumerate(zip(sensor_positions, acceleration)):
    axes[i].plot(time, signal, linewidth=1.5)
    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('Acceleration')
    axes[i].set_title(f'Sensor at Position {sensor_pos:.2f}')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Compare Different Damage Scenarios

Let's compare vibration signatures for different damage conditions.

In [ ]:
# Generate samples for multiple damage scenarios
scenarios = [
    {'level': 0.0, 'location': 0.5, 'name': 'Healthy'},
    {'level': 0.1, 'location': 0.2, 'name': 'Light Root Damage'},
    {'level': 0.2, 'location': 0.5, 'name': 'Moderate Center Damage'},
    {'level': 0.3, 'location': 0.8, 'name': 'Severe Tip Damage'}
]

all_samples = []
for scenario in scenarios:
    print(f"Generating samples for {scenario['name']}...")
    samples = quick_train_and_generate(
        damage_level=scenario['level'],
        damage_location=scenario['location'],
        num_samples=10
    )
    samples['scenario_name'] = scenario['name']
    all_samples.append(samples)

In [ ]:
# Compare RMS amplitudes across scenarios
rms_values = []
scenario_names = []

for samples in all_samples:
    # Calculate RMS for each sample and average
    rms_per_sample = np.sqrt(np.mean(samples['acceleration']**2, axis=(1, 2)))
    rms_values.append(rms_per_sample)
    scenario_names.append(samples['scenario_name'])

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
positions = range(len(scenario_names))

for i, (rms_vals, name) in enumerate(zip(rms_values, scenario_names)):
    ax.boxplot(rms_vals, positions=[i], widths=0.6, 
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    
ax.set_xticks(positions)
ax.set_xticklabels(scenario_names, rotation=45, ha='right')
ax.set_ylabel('RMS Acceleration')
ax.set_title('Vibration Amplitude vs Damage Condition')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Frequency Domain Analysis

Analyze how damage affects the frequency content of vibrations.

In [ ]:
from scipy import signal

# Compute and plot frequency spectra
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Frequency Analysis of Different Damage Conditions', fontsize=16)

sampling_rate = 1000  # Hz

for idx, (samples, ax) in enumerate(zip(all_samples, axes.flat)):
    # Average spectrum across all samples and sensors
    all_spectra = []
    
    for sample_idx in range(samples['acceleration'].shape[0]):
        for sensor_idx in range(samples['acceleration'].shape[1]):
            signal_data = samples['acceleration'][sample_idx, sensor_idx, :]
            frequencies, psd = signal.welch(signal_data, fs=sampling_rate, nperseg=256)
            all_spectra.append(psd)
    
    # Average spectrum
    avg_spectrum = np.mean(all_spectra, axis=0)
    
    # Plot
    ax.semilogy(frequencies, avg_spectrum, linewidth=2)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.set_title(samples['scenario_name'])
    ax.grid(True, alpha=0.3)
    
    # Highlight dominant frequencies
    peaks, _ = signal.find_peaks(avg_spectrum, height=np.max(avg_spectrum)*0.1)
    if len(peaks) > 0:
        peak_freqs = frequencies[peaks]
        peak_amps = avg_spectrum[peaks]
        ax.scatter(peak_freqs, peak_amps, color='red', s=50, alpha=0.7, zorder=5)

plt.tight_layout()
plt.show()

## 4. Model Training Demonstration

Show the training process and monitor convergence.

In [ ]:
# Initialize surrogate model for training demonstration
surrogate = DroneWingSurrogate()

# Configure for shorter training demo
training_config = {
    'epochs': 100,  # Reduced for demo
    'batch_size': 512,
    'learning_rate': 1e-3
}

print("Starting training demo (100 epochs)...")
print("Note: Full training typically uses 1000+ epochs for production models.")

# Train model (this will show progress)
history = surrogate.train(training_config=training_config, verbose=True)

In [ ]:
# Plot training convergence
visualizer.plot_training_history(history)
plt.show()

## 5. Physics Compliance Validation

Verify that our trained model satisfies the underlying physics.

In [ ]:
from evaluation.validation import quick_validation

# Run quick physics validation
print("Validating physics compliance...")
validation_results = quick_validation(surrogate.pinn_model, surrogate.config)

# Display results
print("\nPHYSICS VALIDATION RESULTS:")
print("=" * 40)

for category, metrics in validation_results.items():
    print(f"\n{category.upper()}:\n" + "-" * 20)
    for metric_name, value in metrics.items():
        # Determine pass/fail
        if 'residual' in metric_name:
            status = "✓ PASS" if abs(value) < 1e-3 else "✗ FAIL"
        elif 'stability' in metric_name:
            status = "✓ PASS" if value > 0.95 else "✗ FAIL"
        else:
            status = "✓ PASS"
            
        print(f"  {metric_name:30s}: {value:10.6f} [{status}]")

## 6. Damage Detection Performance

Evaluate the model's ability to distinguish between healthy and damaged states.

In [ ]:
# Generate test data for damage detection evaluation
print("Generating test data for damage detection...")

# Healthy test data
healthy_samples = surrogate.generate_samples(
    damage_level=0.0, damage_location=0.5, num_samples=20
)

# Damaged test data
damaged_samples = surrogate.generate_samples(
    damage_level=0.25, damage_location=0.3, num_samples=20
)

# Extract features (simple RMS-based approach for demo)
healthy_features = np.sqrt(np.mean(healthy_samples['acceleration']**2, axis=(1, 2)))
damaged_features = np.sqrt(np.mean(damaged_samples['acceleration']**2, axis=(1, 2)))

# Simple threshold-based classification
threshold = np.mean([healthy_features.mean(), damaged_features.mean()])

healthy_predictions = (healthy_features > threshold).astype(int)
damaged_predictions = (damaged_features > threshold).astype(int)

# Ground truth labels
y_true = np.concatenate([np.zeros(len(healthy_features)), np.ones(len(damaged_features))])
y_pred = np.concatenate([healthy_predictions, damaged_predictions])

# Compute metrics
metrics = SHMMetrics.compute_classification_metrics(y_true, y_pred)

print("DAMAGE DETECTION PERFORMANCE:")
print("=" * 40)
for metric_name, value in metrics.items():
    print(f"{metric_name:15s}: {value:.4f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Feature distribution
ax1.hist(healthy_features, alpha=0.7, label='Healthy', bins=10, color='blue')
ax1.hist(damaged_features, alpha=0.7, label='Damaged', bins=10, color='red')
ax1.axvline(threshold, color='black', linestyle='--', linewidth=2, 
            label=f'Threshold = {threshold:.4f}')
ax1.set_xlabel('RMS Acceleration')
ax1.set_ylabel('Frequency')
ax1.set_title('Feature Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Confusion matrix visualization
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Healthy', 'Damaged'],
            yticklabels=['Healthy', 'Damaged'],
            ax=ax2)
ax2.set_title('Confusion Matrix')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')

plt.tight_layout()
plt.show()

## 7. Summary and Key Insights

This demonstration showed how Gen-SHM can:

1. **Generate synthetic vibration data** for arbitrary damage scenarios
2. **Capture physics-based relationships** between damage and vibration signatures
3. **Enable damage detection** through learned patterns
4. **Maintain physics compliance** throughout the learning process

### Key Advantages:
- **Data efficiency**: No need for expensive physical testing
- **Physics-grounded**: Built-in physical constraints ensure realistic outputs
- **Zero-shot capability**: Can generalize to unseen damage scenarios
- **Edge deployable**: Lightweight surrogate suitable for real-time applications

### Potential Applications:
- Drone fleet structural health monitoring
- Predictive maintenance scheduling
- Safety-critical damage detection
- Training data augmentation for ML systems